In [1]:
import json
from dataclasses import (
    dataclass,
    field,
    asdict
)

import urllib.request

messages = [
    {
        "role": "system",
        "content":
            "Answer briefly and accurately."
    },
    {
        "role": "user",
        "content":
            "What is 2 + 2?"
    }
]

print(
    json.dumps(
        messages,
        indent=2
    )
)

context_limit = 4096

input_tokens = 3000

reserved_output = 500

print(
    "Remaining budget:",
    context_limit -
    input_tokens -
    reserved_output
)

[
  {
    "role": "system",
    "content": "Answer briefly and accurately."
  },
  {
    "role": "user",
    "content": "What is 2 + 2?"
  }
]
Remaining budget: 596


In [2]:
@dataclass
class Response:

    content: str = ""

    reasoning: str | None = None

    tool_call: dict | None = None

    metadata: dict = field(
        default_factory=dict
    )

example = Response(
    content="4"
)

print(example)

print(example.metadata)

Response(content='4', reasoning=None, tool_call=None, metadata={})
{}


In [4]:
def parse_response(data):

    choices = (
        data.get("choices")
        or []
    )

    if not choices:

        raise ValueError(
            "Endpoint returned no choices"
        )

    message = choices[0]["message"]

    calls = (
        message.get("tool_calls")
        or []
    )

    return Response(
        content=
            message.get("content")
            or "",

        reasoning=
            message.get(
                "reasoning"
            ),

        tool_call=
            calls[0]
            if calls
            else None,

        metadata={
            "model":
                data.get(
                    "model",
                    "unknown"
                ),
            "usage":
                data.get(
                    "usage"
                ) or {}
        }
    )

sample = {
    "choices": [
        {
            "message": {
                "content": "4"
            }
        }
    ]
}

print(
    parse_response(sample)
)

Response(content='4', reasoning=None, tool_call=None, metadata={'model': 'unknown', 'usage': {}})


In [5]:
class FixtureTransport:

    def __init__(self, content):

        self.content = content

    def send(self, messages):

        return {
            "choices": [
                {
                    "message": {
                        "content":
                            self.content
                    }
                }
            ]
        }

transport = FixtureTransport(
    "The answer is 4."
)

reply = transport.send(
    messages
)

print(reply)

{'choices': [{'message': {'content': 'The answer is 4.'}}]}


In [6]:
class LLM:

    def __init__(
        self,
        transport
    ):
        self.transport = transport

    def chat(
        self,
        messages
    ):

        raw = self.transport.send(
            messages
        )

        return parse_response(
            raw
        )

llm = LLM(
    transport
)

print(
    llm.chat(messages)
)

Response(content='The answer is 4.', reasoning=None, tool_call=None, metadata={'model': 'unknown', 'usage': {}})


In [8]:
class TinyAgent:

    def __init__(self, llm):

        self.llm = llm

        self.trajectory = []

    def run(self, user_message):

        messages = [
            {
                "role": "user",
                "content": user_message
            }
        ]

        response = self.llm.chat(
            messages
        )

        self.trajectory.append(
            {
                "input": user_message,
                "output": response.content
            }
        )

        return response

agent = TinyAgent(llm)

result = agent.run(
    "What is 2 + 2?"
)

print(result.content)

The answer is 4.


In [9]:
print(agent.trajectory)


[{'input': 'What is 2 + 2?', 'output': 'The answer is 4.'}]


## Trajectory Logging

The agent stores a history of inputs and outputs.

Trajectory logging allows previous interactions to be reviewed and analyzed.

This information can be useful for debugging and evaluating agent behavior.

## Controlled Change

I changed the fixture response from a simple answer to a longer answer.

Prediction:

A longer response will still be processed correctly because the transport and parsing logic do not depend on the length of the response.

In [10]:
larger_transport = FixtureTransport(
    "The answer is 4. This is a longer response used for testing."
)

larger_llm = LLM(
    larger_transport
)

larger_agent = TinyAgent(
    larger_llm
)

larger_result = larger_agent.run(
    "What is 2 + 2?"
)

print(
    larger_result.content
)

The answer is 4. This is a longer response used for testing.


## Controlled Change Results

The prediction was correct.

The parser, LLM wrapper, and TinyAgent all handled the longer response successfully.

Changing the response length did not require any code modifications.

This suggests that the system can process responses of different lengths without affecting functionality.